# Lab type: review
# Course: ML201 — Applied Machine Learning
# Lesson: Building Production-Ready ML Pipelines
# Task: The code below is correct and working. Read each section, run it, then answer the judgment questions in the markdown cells below each block.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import hashlib
import sklearn
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

np.random.seed(42)
n = 2000

monthly_spend = np.random.lognormal(mean=4.0, sigma=0.6, size=n)
account_age = np.random.exponential(24, n).clip(1, 120)  # months
support_tickets = np.random.poisson(1.5, n).clip(0, 15).astype(float)
region = np.random.choice(['north', 'south', 'east', 'west'], n, p=[0.3, 0.25, 0.25, 0.2])
plan_type = np.random.choice(['basic', 'standard', 'pro'], n, p=[0.4, 0.4, 0.2])

# ~20% churn rate
plan_enc = np.where(plan_type == 'basic', 0.3, np.where(plan_type == 'standard', 0.0, -0.4))
log_odds_churn = (
    -2.5
    - 0.003 * monthly_spend
    - 0.015 * account_age
    + 0.35 * support_tickets
    + plan_enc
    + 0.15 * (region == 'north').astype(float)
)
prob_churn = 1 / (1 + np.exp(-log_odds_churn))
churned = np.random.binomial(1, prob_churn)

# Introduce 5% missing values in monthly_spend and support_tickets
miss_spend_idx = np.random.choice(n, size=int(0.05 * n), replace=False)
miss_tickets_idx = np.random.choice(n, size=int(0.05 * n), replace=False)
monthly_spend[miss_spend_idx] = np.nan
support_tickets[miss_tickets_idx] = np.nan

df = pd.DataFrame({
    'monthly_spend': monthly_spend,
    'account_age': account_age,
    'support_tickets': support_tickets,
    'region': region,
    'plan_type': plan_type,
    'churned': churned
})

print(f"Dataset shape   : {df.shape}")
print(f"Churn rate      : {df['churned'].mean():.2%}")
print("\nMissing values:")
print(df.isnull().sum())

X = df.drop('churned', axis=1)
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Part 1: Pipeline + ColumnTransformer

In [ ]:
numeric_features = ['monthly_spend', 'account_age', 'support_tickets']
categorical_features = ['region', 'plan_type']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))
])

pipe.fit(X_train, y_train)

test_auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
print(f"Test AUC: {test_auc:.3f}")
print(f"\nPipeline steps: {list(pipe.named_steps.keys())}")
print("\nColumnTransformer transformers:")
for name, trans, cols in pipe.named_steps['preprocessor'].transformers_:
    print(f"  '{name}': {type(trans).__name__} -> columns {cols}")

**Question 1:** The pipeline was fitted with `pipe.fit(X_train, y_train)`. If new data arrives at inference time with some missing `monthly_spend` values, will the pipeline handle them correctly? Trace through what each step does.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q1</summary>

**Yes, the pipeline handles missing values correctly at inference time.** When `pipe.predict_proba(X_new)` is called, the `preprocessor` step runs first. Inside the numeric sub-pipeline: `SimpleImputer` (fitted on `X_train`, which stored the median of `monthly_spend` from training data) replaces any NaN values with that stored median; `StandardScaler` (also fitted on `X_train`) then standardises the imputed values using the training mean and standard deviation. The resulting clean array flows into the `RandomForestClassifier`.

The key point: all preprocessing state is frozen from training. There is no re-fitting at inference time — the imputer and scaler have stored parameters from `X_train` that are applied consistently to every new input, including NaN-containing rows.

</details>

**Question 2:** A colleague suggests separating preprocessing from the pipeline: fit the imputer and scaler on `X_train` separately, then create the pipeline with just the model. What goes wrong at deployment when raw data arrives for inference?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q2</summary>

**What goes wrong at deployment:** When raw data arrives at the inference API, someone (or some code) must remember to apply the pre-fitted imputer and scaler before calling the model. These preprocessing objects must be separately serialised and loaded. Any of the following failures will occur silently or noisily: (1) if the imputer step is skipped, the model receives a raw DataFrame with NaN values and either errors or produces wrong predictions; (2) if a downstream service loads only the model artifact, it may pass incorrectly preprocessed arrays; (3) if the preprocessing artifacts are from a different training run, there is a statistical mismatch between what the model expects and what it receives.

The sklearn Pipeline prevents all of this by bundling every step into one artifact — loading the pipeline gives you the complete, stateful preprocessing + model chain.

</details>

## Part 2: Serialise the Full Pipeline

> **Security note — joblib/pickle:** `joblib.dump` and `joblib.load` use Python's pickle protocol, which can execute arbitrary code when deserialising a malicious file. The loads below are safe because the files are written and read within this notebook session — no external or user-supplied bytes are deserialised. **In production, only load `.pkl` files that your own training code produced, stored in an access-controlled location (e.g. an internal artefact store or S3 with restricted IAM policies). Never load a pickle file received from an untrusted source.**

In [ ]:
# --- Common mistake: saving only the model step ---
rf_model_only = pipe.named_steps['model']
joblib.dump(rf_model_only, '/tmp/model_only.pkl')
print("Saved model-only artifact: /tmp/model_only.pkl")

# Safe to load: written two lines above in this same session.
loaded_model_only = joblib.load('/tmp/model_only.pkl')
try:
    preds_bad = loaded_model_only.predict_proba(X_test)[:, 1]
    print("Prediction succeeded (unexpected)")
except Exception as e:
    print(f"\nError when predicting with model-only artifact:")
    print(f"  {type(e).__name__}: {e}")

In [ ]:
# --- Correct pattern: save the full pipeline ---
joblib.dump(pipe, '/tmp/full_pipeline.pkl')

# Safe to load: written one line above in this same session.
loaded_pipe = joblib.load('/tmp/full_pipeline.pkl')
preds = loaded_pipe.predict_proba(X_test)[:, 1]
loaded_auc = roc_auc_score(y_test, preds)
print(f"Pipeline loaded successfully. AUC: {loaded_auc:.3f}")

**Question 3:** The `model_only.pkl` artifact works fine when called with pre-processed numpy arrays. Describe a specific scenario in a production API where this will silently produce wrong predictions rather than raising an error.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q3</summary>

**Scenario for silent wrong predictions:** A new `region` value appears in production data (e.g., a previously unseen geographic area). The `OneHotEncoder` in the full pipeline uses `handle_unknown='ignore'`, which maps unknown categories to all-zeros — the model receives a feature vector with zeros for all region columns and still returns a probability without error. The `model_only.pkl` artifact expects a pre-processed numpy array; if the calling code uses an old preprocessing version that label-encodes the new region as an out-of-range integer (e.g., index 5 when the model was trained on indices 0–3), the model silently receives a shifted feature vector and produces probabilities based on misaligned feature columns — wrong predictions with no error raised.

</details>

**Question 4:** If you update the model 3 months later (refit on new data), what must you serialise? What does NOT need to be re-serialised?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q4</summary>

**Must re-serialise the entire full_pipeline.pkl.** When you refit on new data, the `SimpleImputer` learns new median values (the new training set may have different distributions), the `StandardScaler` learns new means and standard deviations, and the `OneHotEncoder` may encounter new categorical values — all of these preprocessing parameters change. The new model was trained on data preprocessed with the new parameters and expects those parameters at inference time.

**What does NOT need re-serialising:** The Python source code, the pipeline architecture definition, feature engineering logic external to the pipeline, or model monitoring/logging infrastructure. The serialised artifact is purely the fitted state; the code that defines the pipeline structure remains the same.

</details>

## Part 3: Reproducibility

In [ ]:
# Practice 1: Check all stochastic components have random_state
print("=== random_state parameters in the pipeline ===")
all_params = pipe.get_params()
random_state_params = {k: v for k, v in all_params.items() if 'random_state' in k}
for k, v in random_state_params.items():
    print(f"  {k}: {v}")

In [ ]:
# Practice 2: Log metadata
dataset_hash = hashlib.md5(
    pd.util.hash_pandas_object(X_train, index=True).values
).hexdigest()

metadata = {
    'trained_at': datetime.utcnow().isoformat(),
    'n_train': len(X_train),
    'test_auc': round(test_auc, 4),
    'sklearn_version': sklearn.__version__,
    'dataset_hash': dataset_hash
}

print("=== Training metadata ===")
for k, v in metadata.items():
    print(f"  {k}: {v}")

In [ ]:
# Practice 3: Pin requirements (key library versions)
import numpy
print("=== Key library versions ===")
print(f"  scikit-learn : {sklearn.__version__}")
print(f"  numpy        : {numpy.__version__}")
print(f"  pandas       : {pd.__version__}")
print("\nAdd these to requirements.txt or pyproject.toml to reproduce this environment.")

**Question 5:** If scikit-learn upgrades from 1.4 to 1.5 and the default value of a hyperparameter changes, your saved `pipeline.pkl` will still use the 1.4 default at load time. But what breaks when you try to retrain on new data using the same code without pinned versions?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q5</summary>

**What breaks when retraining without pinned versions:** The retraining code will use scikit-learn 1.5 defaults for any hyperparameter that changed (e.g., a different default `max_features` or a changed imputation strategy). The retrained model uses different settings than the one currently in production, making them incompatible for direct comparison in A/B testing, performance monitoring, or rollback. The loaded artifact (using 1.4 defaults frozen at serialisation time) and the retrained model (using 1.5 defaults) represent different experiments — you have no way to attribute performance differences to the model refit versus the library update.

</details>

**Question 6:** The `dataset_hash` is computed from `X_train`. What scenario would cause this hash to be the same even though the model's performance has changed significantly?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q6</summary>

**The hash would be the same if only the labels changed.** The hash is computed from `X_train` (the feature matrix) alone. If you corrected mislabelled examples in `y_train`, relabelled fraud cases, or changed the target definition (e.g., from 30-day to 60-day churn), the features would be identical and the hash would match the previous run — but the model learns a different decision boundary and produces different performance. The hash would also be the same if rows were reordered without adding or removing data (though `hash_pandas_object` is row-order-sensitive), or if only a non-feature column changed.

</details>

## Part 4: Pipeline Test

In [ ]:
def test_pipeline():
    # Safe to load: /tmp/full_pipeline.pkl was written by this notebook's Part 2 cell.
    loaded = joblib.load('/tmp/full_pipeline.pkl')
    test_row = pd.DataFrame([{
        'monthly_spend': 75.0,
        'account_age': 18,
        'support_tickets': 3,
        'region': 'west',
        'plan_type': 'standard'
    }])
    prob = loaded.predict_proba(test_row)[0, 1]
    assert 0.0 <= prob <= 1.0, f"Probability out of range: {prob}"
    print(f"Test passed. Predicted churn probability: {prob:.3f}")

test_pipeline()

**Question 7:** This test uses a hard-coded row with specific values. What three things does it verify? What would it NOT catch?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q7</summary>

**What the test verifies:** (1) The pipeline artifact can be deserialised without error — no missing dependencies or incompatible pickle format; (2) the pipeline accepts a raw DataFrame with the expected column names and types without raising a schema error; (3) the output is a valid probability scalar in the range [0, 1].

**What it does NOT catch:** Schema drift — a new column added to production data or a renamed column shifts feature positions silently; edge cases — all-NaN rows, negative values in `monthly_spend`, or a `region` category not seen during training; distribution shift — the probability could be 0.42 now and 0.95 after model decay, both in range but one meaningless; silent type coercion — if the DataFrame column types differ from training (e.g., `account_age` passed as string), the pipeline may coerce silently and produce wrong results.

</details>

**Question 8:** In a CI/CD pipeline, where should this test run — before or after deployment? What events should trigger a re-run?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Q8</summary>

**Before deployment** — this is a pre-deployment gate. Running it after deployment means a broken artifact has already reached production and could be serving wrong predictions to real users. The test should block the deployment pipeline if it fails.

**Events that should trigger a re-run:** (1) any new model training run (new artifact written); (2) any change to the preprocessing code, feature schema, or pipeline definition; (3) any library version upgrade (scikit-learn, joblib, pandas); (4) any change to the inference API code that handles the pipeline artifact; (5) as a periodic health check in production to catch runtime environment drift (library upgrades pushed by infrastructure teams without a code change).

</details>

## Summary

Answer these final check questions in one sentence each:

1. Name two components that belong inside the sklearn `Pipeline` and one component that should NOT be inside it (and why).

2. You serialised the full pipeline, but a colleague re-fits only the model step and replaces `pipe.named_steps['model']` before saving. What can go wrong when the updated artifact is loaded in production?

3. What are the three elements of the reproducibility triad (what to log, what to pin, what to fix), and which one do developers most commonly skip?

4. A hard-coded "smoke test" on one row passes in CI. Give one example of a real-world input that would expose a bug the smoke test does not cover.

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Inside vs outside the Pipeline:** Preprocessors (imputer, scaler, encoder) and the model step belong inside — they have fitted state that must be serialised and applied consistently. Raw data I/O (reading from S3, parsing JSON, database queries) should NOT be inside the Pipeline because sklearn Pipelines are not designed to handle external I/O, stateful connections, or non-array operations, and including them breaks serialisation, portability, and unit testability.

2. **Replacing only the model step:** If a colleague refits the model outside the pipeline on data with a different scale or schema — or trains it after applying different preprocessing — the serialised artifact will pass data through the original preprocessing (fitted on the old training set) into a model that expects differently scaled or shaped input. The predictions will be wrong, there will be no error raised, and the bug will be invisible unless output values are monitored.

3. **The three reproducibility elements:** (1) Log metadata (training date, dataset hash, hyperparameters, performance metrics), (2) pin library versions (requirements.txt or pyproject.toml), and (3) fix random seeds (`random_state` in all stochastic components) — developers most commonly skip fixing random seeds, meaning the model artifact differs from run to run even on the same data and code, making debugging and comparison impossible.

4. **Smoke test blind spot:** A row where `region` contains a value not seen during training (e.g., `'pacific'`) would be silently mapped to all-zeros by `handle_unknown='ignore'` — the pipeline returns a probability in [0,1] with no error, but the prediction is computed with the entire region encoding zeroed out, which the hard-coded smoke test row (using a known training region like `'west'`) would never expose.

</details>